In [384]:
from manim import *

In [391]:

def DerivationOfRdrdthetaGraph(self):
  speed = 0.5

  axes = Axes(
    x_range=[-1, 5],
    y_range=[-1, 5],
    x_length=6,
    y_length=6,
    axis_config={"color": WHITE, "include_tip": False}
  )
  axes.shift(LEFT * 3.5 + UP * 0.8)
  origin = axes.get_origin()

  self.play(FadeIn(axes), run_time = speed)
  self.wait()

  # Parameters
  R = 4
  dr = 1
  theta = PI / 6
  dtheta = PI / 4

  # Helper function to calculate polar to cartesian
  def get_point(r, theta):
    return origin + np.array([r * np.cos(theta), r * np.sin(theta), 0])


  # Get the points for the sector (curvy rectangle)
  p1 = get_point(R, theta)
  p2 = get_point(R + dr, theta)
  p3 = get_point(R + dr, theta + dtheta)
  p4 = get_point(R, theta + dtheta)

  # Make initial line with radius r and theta
  line_r = Line(origin, p1, color=BLUE)
  label_r = MathTex("r").next_to(line_r, DOWN, buff=0.3).shift(1.5 *UP + RIGHT)


  self.play(Create(line_r), Write(label_r))

  # Extend the radius a little bit by dr
  line_dr = DashedLine(p1, p2, color = BLUE_A)
  label_dr = MathTex("dr").next_to(line_dr, DOWN, buff = 0.15).shift(RIGHT * 0.15 + UP * 0.2)

  self.play(Create(line_dr), Write(label_dr))
  self.wait()

  # Copy of original line to rotate into new line with dtheta + theta
  line_dr_copy = line_dr.copy()
  line_r_copy = line_r.copy()
  line_r_copy.stroke_color = GRAY
  
  outer_arc = Arc(
    radius = R + dr,
    start_angle = theta,
    angle = dtheta,
    color = PURPLE,
  ).shift(origin)

  label_dtheta = MathTex("d\\theta").move_to(
    origin + (R + dr + 0.5) * np.array([np.cos(theta + dtheta)/4, np.sin(theta + dtheta)/4, 0])
  ).shift(RIGHT * 0.4 + DOWN * 0.3)

  reference_angle = Arc(
    radius = 0.8,
    start_angle=theta,
    angle = dtheta,
    color = GREEN
  ).shift(origin)

  sector = AnnularSector(
    outer_radius=R + dr,
    inner_radius=R,
    start_angle=theta,
    angle=dtheta,
    color=RED,
    fill_opacity=0.3,
  ).shift(origin)

  arc_inner = Arc(
    radius = R,
    start_angle = theta,
    angle = dtheta,
    color = PURPLE,
  ).shift(origin)
  label_arc = MathTex("r \\cdot d\\theta").scale(0.9).next_to(arc_inner, buff = 0.1).shift(LEFT * 1.9 + DOWN * 0.2)

  dA = MathTex("dA").next_to(arc_inner, UP, buff=0.2).shift(DOWN * 0.65 + RIGHT * 0.45)

  self.play(
    Rotate(line_dr_copy, angle = dtheta, about_point=origin),
    Write(label_dtheta),
    FadeIn(reference_angle),
    Rotate(line_r_copy, angle = dtheta, about_point=origin), 
    Create(outer_arc),
    Create(arc_inner),
    Write(label_arc),
    run_time = 1
  )
  self.play(
    FadeIn(sector, dA),
    run_time = 1
  )

  rect = Rectangle(
      width=3,
      height=5,
      stroke_width = 2,
      color = RED,
      fill_opacity = 0.3,
  ).to_edge(RIGHT, buff=1.5).shift(UP * 0.5)

  dA2 = MathTex("dA").move_to(rect.get_center())
  dr = MathTex("dr").move_to(rect.get_bottom() + DOWN)
  rdtheta = MathTex("r \\cdot d\\theta").move_to(rect.get_left() + LEFT * 1.2)

  area = MathTex("Area &= length \\times width").next_to(rect, DOWN * 3, buff = 0.5)
  f1 = MathTex("dA &= rd\\theta \\cdot dr").next_to(rect, DOWN * 3, buff = 0.5)
  f2 = MathTex("dA &= rdrd\\theta").next_to(rect, DOWN * 3, buff = 0.5)
  dxdy = MathTex("dxdy &= rdrd\\theta").next_to(rect, DOWN * 3, buff = 0.5)

  left_brace = Brace(
    rect,
    LEFT,
    color = BLUE_E,
  )
  bottom_brace = Brace(
    rect,
    DOWN,
    color = WHITE,
  )

  self.play(Transform(sector, rect), run_time = 0.5)
  self.play(Write(dA2), run_time = 0.5)
  self.play(
    Create(bottom_brace),
    Write(dr),
    run_time = 1
  )
  self.play(
    Create(left_brace),
    Write(rdtheta),
    run_time = 1
  )
  area_speed = 1.5
  self.play(Write(area), run_time = area_speed)
  self.play(ReplacementTransform(area, f1), run_time = area_speed)
  self.play(ReplacementTransform(f1, f2), run_time = area_speed)
  self.play(ReplacementTransform(f2, dxdy), run_time = area_speed)
  self.wait()
  
  items = VGroup(axes, line_r, line_dr, line_dr_copy, dA2, line_r_copy, arc_inner, outer_arc, sector, rdtheta, dr, reference_angle, label_dtheta, label_dr, label_arc, label_r, dA, left_brace, bottom_brace, dxdy)

  self.play(
    FadeOut(items), run_time = 1.5
  )

def DerivationRectToPolarProof(self):
  DerivationOfRdrdthetaGraph(self)

In [392]:
class GlowDot(Dot3D):
  def __init__(self, point=ORIGIN, color=WHITE, radius=0.1, glow_radius=0.3, layers=15, **kwargs):
    kwargs.pop('renderer', None)
    kwargs.pop('scene', None)
    
    super().__init__(radius=radius, color=color, **kwargs)

    self.glow = VGroup()
    for i in range(layers):
      glow_sphere = Sphere(
        radius=radius + i * glow_radius / layers,
        color=color,
        resolution=(10, 10)
      ).move_to(point)
      opacity = 0.4 * (1 - i / layers) ** 3  # Cubic falloff
      glow_sphere.set_opacity(opacity)
      glow_sphere.set_stroke(width=0) 
      self.glow.add(glow_sphere)

    self.add_to_back(self.glow)
    self.move_to(point)

  def set_glow_color(self, color):
    self.set_color(color)
    self.glow.set_color(color)
    return self

  def move_to(self, point):
    super().move_to(point)
    self.glow.move_to(point)
    return self

In [396]:
def DeriveCircleFormula(self):
  speed = 0.5

  axes = Axes(
    x_range = [-3, 3, 1],
    y_range = [-3, 3, 1],
    x_length = 7.5,
    y_length = 7.5,
    axis_config={"include_tip": False}
  ).shift(LEFT * 3)
  
  origin = axes.c2p(0, 0)
  R = 3.25
  TH = PI/4

  circle = Circle(
    radius=R,
    color=BLUE_E,
    stroke_width=3,
    fill_opacity = 0
  ).move_to(origin)

  def get_point(r, theta):
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    return origin + np.array([x, y, 0])

  dot = GlowDot(radius=0.01, glow_radius=0.2, color=PURPLE).move_to(origin)
  dot.set_glow_color(PURPLE)

  point_circle = get_point(R, TH)
  radius_line = Line(
    start=origin,
    end=get_point(R, TH),
    color=WHITE,
    stroke_width=2
  )
  x_line = Line(
    start = origin,
    end = np.array([point_circle[0], 0, 0]),
    stroke_width = 3,
    color = RED,
  )
  y_line = Line(
    start = np.array([point_circle[0], 0, 0]),
    end = point_circle,
    stroke_width = 3,
    color = RED,
  )

  x = MathTex("x", font_size = 32).next_to(x_line, DOWN, buff = 0.2)
  y = MathTex("y", font_size = 32).next_to(y_line, RIGHT, buff = 0.2)
  r = MathTex("r", font_size = 32).move_to(radius_line).shift(UP * 0.3)
  xypoint = MathTex("(x,y)", font_size = 32).next_to(y_line, UR, buff = 0.2)
  a = MathTex("a^2", font_size = 40).next_to(axes.get_right(), buff=2)
  plus = MathTex("+", font_size = 40).next_to(a.get_right()).shift(DOWN * 0.05)
  b = MathTex("b^2", font_size = 40).next_to(plus.get_right())
  eq = MathTex("&=", font_size = 40).next_to(b.get_right()).shift(DOWN * 0.05)
  c = MathTex("c^2", font_size = 40).next_to(eq.get_right()).shift(UP * 0.1)
  
  pythag = VGroup(a, plus, b, eq, c)

  x_sq = MathTex("x^2", font_size = 40).next_to(axes.get_right(), buff=2)
  y_sq = MathTex("y^2", font_size = 40).next_to(plus.get_right())
  r_sq = MathTex("r^2", font_size = 40).next_to(eq.get_right()).shift(UP * 0.1)


  # Animations
  self.play(FadeIn(axes), run_time = speed)
  self.play(Create(circle), Create(dot), run_time = speed)
  self.play(dot.animate.move_to(get_point(R, TH)), run_time = speed)
  self.play(Create(radius_line), run_time = speed)
  self.play(Create(x_line), Create(y_line), run_time = speed)
  self.play(Write(r), Write(x), Write(y), Write(xypoint), run_time = speed)
  self.play(Write(pythag), run_time = speed + 1)
  self.play(ReplacementTransform(a, x_sq), ReplacementTransform(b, y_sq), ReplacementTransform(c, r_sq), run_time = speed)
  self.wait()
  self.play(FadeOut(x_sq, y_sq, r_sq, eq, plus, axes, radius_line, x_line, y_line, circle, xypoint, x, y, r, dot), run_time=speed)

In [403]:
class GaussianIntegral(Scene):
  def construct(self):
    font_size = 40
    speed = 0.5

    # Make group for latex strings for looping through later
    equations = [
        MathTex(R"\int_{-\infty}^{+\infty} e^{-x^2} dx", font_size=font_size),
        MathTex(R"I &= \int_{-\infty}^{+\infty} e^{-x^2} dx", font_size=font_size),
        MathTex(R"I^2 &= \left( \int_{-\infty}^{+\infty} e^{-x^2} dx \right)^2", font_size=font_size),
        MathTex(R"I^2 &= \int_{-\infty}^{+\infty} e^{-x^2} dx \cdot \int_{-\infty}^{+\infty} e^{-y^2} dy", font_size=font_size),
        MathTex(R"I^2 &= \int_{-\infty}^{+\infty} \int_{-\infty}^{+\infty} e^{-x^2} \cdot e^{-y^2} dxdy", font_size=font_size),
        MathTex(R"I^2 &= \int_{-\infty}^{+\infty} \int_{-\infty}^{+\infty} e^{-x^2 - y^2} dxdy", font_size=font_size),
        MathTex(R"I^2 &= \int_{-\infty}^{+\infty} \int_{-\infty}^{+\infty} e^{-(x^2 + y^2)} dxdy", font_size=font_size),
        MathTex(R"\int_{-\infty}^{+\infty} \int_{-\infty}^{+\infty} e^{-(x^2 + y^2)} dxdy", font_size=font_size),
        MathTex(R"\int_{-\infty}^{+\infty} \int_{-\infty}^{+\infty}", font_size=font_size),
        MathTex(R"\int_{-\infty}^{+\infty} \int_{0}^{+\infty}", font_size=font_size),
        MathTex(R"\int_{0}^{2\pi} \int_{0}^{+\infty}", font_size=font_size),
        MathTex(R"\int_{0}^{2\pi} \int_{0}^{+\infty} e^{-(x^2 + y^2)} dxdy", font_size=font_size),
        MathTex(R"\int_{0}^{2\pi} \int_{0}^{+\infty} e^{-(x^2 + y^2)} \cdot rdrd\theta", font_size=font_size),
        MathTex(R"\int_{0}^{2\pi} \int_{0}^{+\infty} e^{-r^2} rdrd\theta", font_size=font_size),
        MathTex(R"-\frac{1}{2} \int_{0}^{2\pi} \Big[ e^{-r^2} \Big]_{0}^{+\infty} d\theta", font_size=font_size),
        MathTex(R"-\frac{1}{2} \int_{0}^{2\pi} \Big[ \lim_{n\to\infty} e^{-r^2} \Big] - e^{-(0)^2} d\theta", font_size=font_size),
        MathTex(R"-\frac{1}{2} \int_{0}^{2\pi} -1 d\theta", font_size=font_size),
        MathTex(R"\frac{1}{2} \int_{0}^{2\pi} d\theta", font_size=font_size),
        MathTex(R"\frac{1}{2} [\theta]_{0}^{2\pi}", font_size=font_size),
        MathTex(R"\frac{1}{2} [2\pi - 0]", font_size=font_size),
        MathTex(R"\frac{2\pi}{2}", font_size=font_size),
        MathTex(R"\pi", font_size=font_size),
        MathTex(R"I^2 &= \pi", font_size=font_size),
        MathTex(R"I &= \sqrt{\pi}", font_size=font_size),
        MathTex(R"\sqrt{\pi}.", font_size=font_size),
    ]
    
    for equ in equations: 
      equ.center()

    # Start with gaussian integral    
    self.play(Write(equations[0]))
    self.wait(speed)

    # Start of derivation animations
    for i in range(6):
      self.play(ReplacementTransform(equations[i], equations[i + 1]), run_time = speed)
      self.wait()

    # Change bounds to polar
    equations[7].to_corner(UP)

    integrals = MathTex(
      R"\int_{-\infty}^{+\infty} \int_{-\infty}^{+\infty}",
      font_size=40,
      tex_to_color_map={
          "-\\infty": PURPLE_A,
          "+\\infty": PURPLE_A,
      }
    )

    # Position to match original equation location
    integrals.move_to(equations[7].get_center()).shift(LEFT * 1.35)

    rect = SurroundingRectangle(
      integrals, 
      stroke_color=PURPLE_A,
      fill_opacity=0,
      stroke_width=3,
      buff=0.1 
    )
    
    rect_axes = Axes(
      x_range=(-3, 3, 2),
      y_range=(0, 1, 1),
      x_length=6,
      y_length=4,
      axis_config={"include_tip": False}
    )
    rect_axes.scale(0.8).to_corner(LEFT, buff=0.2).shift(DOWN * 0.4)

    rect_x_highlight = rect_axes.get_x_axis().copy().set_color(RED).set_stroke(width=5)

    curve = rect_axes.plot(
      lambda x: np.exp(-x**2),
      x_range = (-3, 3, 0.01),
      color = BLUE_C,
      stroke_width = 3
    )

    area_curve = rect_axes.get_area(
      curve,
      x_range=[-3, 3],
      color=PURPLE_E,
      fill_opacity=0.5
    )
    
    polar_axes = PolarPlane(
      size=5,
      radius_max=3
    )
    polar_axes.scale(0.8).to_corner(RIGHT, buff=0.2).shift(DOWN * 0.4 + LEFT * 0.6)

    polar_x_highlight = Line(
      start=polar_axes.get_center(),
      end=polar_axes.get_center() + RIGHT * 2,
      color=RED,
      stroke_width=5
    )

    polar_sector = Sector(
      radius=2,
      start_angle=0,
      angle=2 * PI,
      fill_color=PURPLE_A,
      fill_opacity=0.3,
      stroke_width=0,
      arc_center=polar_axes.get_center(),
    )

    arrow = Arrow( 
      start=rect_axes.get_right() + DOWN * 0.2,
      end=polar_axes.get_left() + DOWN * 0.2,
      buff=speed,
      color=WHITE,
      stroke_width=3
    )

    dxdy = MathTex(R"dxdy", font_size=font_size)
    surround_dxdy = SurroundingRectangle(
      dxdy,
      fill_opacity = 0,
      stroke_width=3,
      stroke_color=PURPLE_E,
      buff=0.1
    )
    surround_dxdy.to_corner(DOWN).shift(RIGHT * 1.825 + UP * 0.25).scale(0.9)

    equations[12].to_edge(UR, buff=0.3)
    f = MathTex(R"e^{-(x^2 + y^2)}", font_size = font_size).move_to(equations[12].get_center()).shift(RIGHT * 0.3 + UP * 0.1)
    surround_f = SurroundingRectangle(
      f,
      fill_opacity = 0,
      stroke_width = 3,
      stroke_color = PURPLE_E,
      buff = 0.1
    )
    
    # ANIMATIONS (continued)
    graph_speed = 1.5
    self.play(equations[6].animate.to_edge(UP), run_time = speed)
    self.play(TransformMatchingTex(equations[6], equations[7]), run_time = speed)
    self.play(FadeIn(rect_axes), FadeIn(polar_axes), Create(arrow), run_time = speed)
    self.play(Create(curve), run_time = speed)
    self.play(
      Create(rect),
      run_time = graph_speed
    )
    self.play(Write(equations[8], run_time=speed))
    self.wait()
    self.play(
      Create(rect_x_highlight), 
      Create(polar_x_highlight),
      TransformMatchingTex(equations[8], equations[9], run_time=speed),
      run_time = graph_speed
    )
    self.wait()
    self.play(
      FadeIn(area_curve),
      Create(polar_sector),
      TransformMatchingTex(equations[9], equations[10], run_time=speed),
      run_time = graph_speed,
      rate_func = smooth
    )
    self.wait()
    self.play(FadeOut(rect_axes, curve, rect_x_highlight, area_curve, polar_axes, polar_sector, polar_x_highlight, arrow, rect, equations[7]), run_time = speed)
    self.play(TransformMatchingTex(equations[10], equations[11]), run_time = speed)    
    self.play(equations[11].animate.to_corner(DOWN))
    self.play(Create(surround_dxdy), run_time=speed)
    
    # Derivation of rdrdtheta
    DerivationRectToPolarProof(self)
    self.play(FadeOut(surround_dxdy), run_time = speed / 2)
    self.play(ReplacementTransform(equations[11], equations[12]))
    self.play(Create(surround_f), run_time = speed)

    # Derivation for circle formula
    DeriveCircleFormula(self)
    self.play(FadeOut(surround_f), run_time = speed)
    self.play(equations[12].animate.center(), run_time = speed)
    self.play(ReplacementTransform(equations[12], equations[13]), run_time = speed)
    self.wait()

    current = 13
    for i in range(len(equations) - current - 1):
      self.play(ReplacementTransform(equations[current + i], equations[current + i + 1]), run_time = speed)
      self.wait()







In [404]:
manim -pqh GaussianIntegral

Manim Community v0.20.0

[03/03/26 03:55:19] INFO     Writing \special{dvisvgm:raw <g                                ]8;id=322427;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\utils\tex_file_writing.py\tex_file_writing.py]8;;\:]8;id=885343;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\utils\tex_file_writing.py#110\110]8;;\
                             id='unique000'>}\sqrt{\pi}.\special{dvisvgm:raw </g>} to                              
                             media\Tex\06f9736fe03af31e.tex                                                        

[03/03/26 03:55:21] INFO     Animation 0 : Partial movie file written in                   ]8;id=650022;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=11719;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene_file_writer.py#590\590]8;;\
                             'C:\Users\dragi\Documents\manimations\code\media\videos\code\                         
                             1080p60\partial_movie_files\GaussianIntegral\2257293077_42062                         
                             74612_223132457.mp4'                                                                  

[03/03/26 03:55:22] INFO     Animation 1 : Partial movie file written in                   ]8;id=18957;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=444697;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene_file_writer.py#590\590]8;;\
                             'C:\Users\dragi\Documents\manimations\code\media\videos\code\                         
                             1080p60\partial_movie_files\GaussianIntegral\2135761390_25744                         
                             73371_1787083701.mp4'                                                                 

                    INFO     Animation 2 : Partial movie file written in                   ]8;id=601548;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=627036;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene_file_writer.py#590\590]8;;\
                             'C:\Users\dragi\Documents\manimations\code\media\videos\code\                         
                             1080p60\partial_movie_files\GaussianIntegral\2135761390_30659                         
                             04809_192510965.mp4'                                                                  

...

[03/03/26 03:57:01] INFO     Animation 84 : Partial movie file written in                  ]8;id=512734;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=948494;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene_file_writer.py#590\590]8;;\
                             'C:\Users\dragi\Documents\manimations\code\media\videos\code\                         
                             1080p60\partial_movie_files\GaussianIntegral\2135761390_26889                         
                             65532_165273749.mp4'                                                                  

                    INFO     Combining to Movie file.                                      ]8;id=624234;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=373767;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene_file_writer.py#742\742]8;;\

[03/03/26 03:57:02] INFO                                                                   ]8;id=832332;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=692291;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene_file_writer.py#893\893]8;;\
                             File ready at                                                                         
                             'C:\Users\dragi\Documents\manimations\code\media\videos\code\                         
                             1080p60\GaussianIntegral.mp4'                                                         
                                                                                                                   

                    INFO     Rendered GaussianIntegral                                                 ]8;id=232571;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene.py\scene.py]8;;\:]8;id=648890;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene.py#278\278]8;;\
                             Played 85 animations                                                                  

                    INFO     Previewed File at:                                                     ]8;id=399515;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\utils\file_ops.py\file_ops.py]8;;\:]8;id=199550;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\utils\file_ops.py#236\236]8;;\
                             'C:\Users\dragi\Documents\manimations\code\media\videos\code\1080p60\G                
                             aussianIntegral.mp4'                                                                  